In [ ]:
!pip install -U transformers datasets accelerate peft trl bitsandbytes sentencepiece protobuf peft

In [ ]:
# transformers  -> load model/tokenizer
# datasets      -> load JSONL dataset
# peft          -> LoRA fine-tuning
# trl           -> SFTTrainer
# bitsandbytes  -> 4-bit QLoRA training
# accelerate    -> device management

In [2]:
import torch
import platform

print("Python platform:", platform.platform())
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("No CUDA GPU found. Fine-tuning will be very slow on CPU.")

Python platform: Windows-10-10.0.26200-SP0
Torch version: 2.7.1+cpu
CUDA available: False
No CUDA GPU found. Fine-tuning will be very slow on CPU.


In [7]:
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

OUTPUT_DIR = "./number-pattern-lora"
MERGED_DIR = "./number-pattern-merged"

print("Base model:", BASE_MODEL)
print("Adapter output:", OUTPUT_DIR)

Base model: Qwen/Qwen2.5-0.5B-Instruct
Adapter output: ./number-pattern-lora


In [8]:
import json
from pathlib import Path

dataset_path = Path("number_finetune_data.jsonl")

def create_example(n):
    parity = "Even" if n % 2 == 0 else "Odd"
    double = n * 2
    square = n * n

    prompt = f"Tell me about number {n}."

    completion = f"""Number: {n}
Parity: {parity}
Double: {double}
Square: {square}
Simple explanation: {n} is an {parity.lower()} number. Its double is {double} and its square is {square}."""

    return {
        "prompt": prompt,
        "completion": completion
    }

examples = [create_example(n) for n in range(1, 201)]

with dataset_path.open("w", encoding="utf-8") as f:
    for ex in examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print("Dataset saved:", dataset_path)
print("Total examples:", len(examples))
print("Sample example:")
print(examples[6])

Dataset saved: number_finetune_data.jsonl
Total examples: 200
Sample example:
{'prompt': 'Tell me about number 7.', 'completion': 'Number: 7\nParity: Odd\nDouble: 14\nSquare: 49\nSimple explanation: 7 is an odd number. Its double is 14 and its square is 49.'}


In [9]:
from datasets import load_dataset

dataset = load_dataset("json", data_files=str(dataset_path), split="train")

print(dataset)
print(dataset[0])

C:\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 200 examples [00:00, 9051.15 examples/s]

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 200
})
{'prompt': 'Tell me about number 1.', 'completion': 'Number: 1\nParity: Odd\nDouble: 2\nSquare: 1\nSimple explanation: 1 is an odd number. Its double is 2 and its square is 1.'}


In [10]:
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("Train examples:", len(train_dataset))
print("Eval examples:", len(eval_dataset))

Train examples: 180
Eval examples: 20


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

use_cuda = torch.cuda.is_available()

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if use_cuda:
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float32,
        trust_remote_code=True
    )

print("Model loaded successfully.")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Model loaded successfully.


In [12]:
def generate_answer(prompt, max_new_tokens=150):
    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt")

    if use_cuda:
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            top_p=0.9,
            do_sample=True
        )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_output

print(generate_answer("Tell me about number 17."))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Tell me about number 17.
assistant
Number 17 is a significant and intriguing figure in various cultures around the world. Here's an overview of its significance:

1. Historical Significance:
   - In many ancient civilizations, numbers were considered sacred or divine.
   - The number 17 was often associated with the number seven, which represented completeness and perfection.

2. Cultural References:
   - In some cultures, 17 is seen as a lucky number, symbolizing good fortune or success.
   - It's also used to represent the number of days in a week (seven).

3. Mathematical Properties:
   - 17 is a prime number, meaning it has no divisors other than 1 and itself.
   - It's the smallest prime number that can


In [ ]:
!pip install peft

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

print(lora_config)

In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    num_train_epochs=5,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=1e-4,
    warmup_ratio=0.03,

    logging_steps=5,

    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,

    eval_strategy="steps",
    eval_steps=50,

    max_length=256,

    completion_only_loss=True,

    fp16=True if use_cuda else False,
    bf16=True if use_cuda and torch.cuda.is_bf16_supported() else False,

    optim="paged_adamw_8bit" if use_cuda else "adamw_torch",

    report_to="none"
)

print(training_args)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    processing_class=tokenizer
)

trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("LoRA adapter saved at:", OUTPUT_DIR)

In [ ]:
print(generate_answer("Tell me about number 17."))

In [ ]:
print(generate_answer("Tell me about number 245."))

In [ ]:
test_prompts = [
    "Tell me about number 3.",
    "Tell me about number 12.",
    "Tell me about number 41.",
    "Tell me about number 99.",
    "Tell me about number 245."
]

for prompt in test_prompts:
    print("=" * 60)
    print("PROMPT:", prompt)
    print(generate_answer(prompt, max_new_tokens=120))

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    OUTPUT_DIR
)

model = fine_tuned_model

print("Fine-tuned adapter loaded.")

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

merge_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

base_model_for_merge = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=merge_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

merged_model = PeftModel.from_pretrained(
    base_model_for_merge,
    OUTPUT_DIR
)

merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained(
    MERGED_DIR,
    safe_serialization=True
)

tokenizer.save_pretrained(MERGED_DIR)

print("Merged model saved at:", MERGED_DIR)

In [ ]:
%%writefile Modelfile
FROM qwen2.5:0.5b
ADAPTER ./number-pattern-lora

PARAMETER temperature 0.2
PARAMETER top_p 0.9
PARAMETER num_ctx 2048

SYSTEM """
You answer number questions in a fixed format:
Number:
Parity:
Double:
Square:
Simple explanation:
"""

In [ ]:
!ollama create number-ft -f Modelfile

In [ ]:
!ollama run number-ft "Tell me about number 21."